# Dataset Groups

This notebook groups [GIFT-Eval](https://github.com/SalesforceAIResearch/gift-eval) datasets based on the models proposed in the [Ensemble TEMPO slides](https://docs.google.com/presentation/d/1GxL_qUQKizv5C_RPzxYiJjxJSPD6-81x1VPOxxYXAl0/edit?slide=id.g35d6bf22e47_0_10#slide=id.g35d6bf22e47_0_10).
- First, we'll group pretraining datasets
- Then, we'll group train-test datasets

Ensemble TEMPO will have models at three different granularities:
- **General** 
  - Three models across three forecasting terms (short, medium, and long)
- **Domain-specific** 
  - 15 models across seven domains and three forecasting terms
- **Dataset-specific** 
  - 97+ models across each dataset
  - **Note:**: Dataset-specific models will only be trained on the train-test split

## Pretraining Split

Load the pretraining split's metadata.

In [77]:
import pandas as pd
from pathlib import Path

split = "pretrain"
metadata_path = Path("resources") / split / "metadata.csv"

df = pd.read_csv(metadata_path)

print(f"Number of unique {split} datasets: {df['name'].nunique()}")
print(f"Number of unique {split} terms: {df['term'].nunique()}")
print(f"Total number of {split} datasets: {len(df)}")
df.head()

Number of unique pretrain datasets: 152
Number of unique pretrain terms: 3
Total number of pretrain datasets: 456


,name,term,freq,domain,num_series,target_dim,_min_series_length,sum_series_length,prediction_length,windows
0,bull,short,H,Energy,41,1,17544,719304,48,20
1,bull,medium,H,Energy,41,1,17544,719304,480,4
2,bull,long,H,Energy,41,1,17544,719304,720,3
3,cmip6_1885,short,6H,Climate,434176,53,7300,3169484800,48,16
4,cmip6_1885,medium,6H,Climate,434176,53,7300,3169484800,480,2


Create a list to store each model's dataset configuration. Later, this list will be used to create a YAML dump.

In [78]:
data = []

### General

View all of the unique forecasting terms and count the number of datasets that belong to each term. Each forecasting term multiplies the original prediciton length by a given multipler:
- "long" multiplies the original prediction length by 15
- "medium" multiplies the original prediction length by 10  
- "short" multiplies the original prediction length by 1 (no change)

In [79]:
def get_count_df(
    df: pd.DataFrame,
    columns: list[str],
    ascending: bool = False,
) -> pd.DataFrame:
    """
    Counts the number of unique combinations of all unique values in the
    specified columns of the given a DataFrame.

    Args:
        df (pd.DataFrame): The input DataFrame.
        columns (list[str]): A list of column names to group by.
        ascending (bool, optional): If True, sort the result in ascending order
            of count. Defaults to False (descending order).

    Returns:
        pd.DataFrame: A DataFrame with the grouped columns and a 'count'
            column, sorted by count.
    """
    count_df = df.groupby(columns).size().reset_index(name="count")
    return count_df.sort_values(by="count", ascending=ascending).reset_index(drop=True)


term_counts = get_count_df(df, columns=["term"], ascending=False)

print(f"Number of unique terms: {len(term_counts)}")
display(term_counts)

Number of unique terms: 3


,term,count
0,long,152
1,medium,152
2,short,152


For each term, add all of the dataset names and the given term to our YAML data and save the resulting YAML file.

In [80]:
import yaml

num_general_configs = 0
for term in df["term"].unique():
    num_general_configs += 1
    names = df[df["term"] == term]["name"].tolist()
    data.append({"term": term, "names": names})

yaml_path = Path("resources") / split / "dataset_configs.yaml"

print(f"Number of general configurations: {num_general_configs}")
with open(yaml_path, "w") as file:
    yaml.dump(data, file, sort_keys=False)

Number of general configurations: 3


Load the YAML file to ensure the configurations were saved correctly.

In [81]:
with open(yaml_path, "r") as file:
    configs = yaml.safe_load(file)

print(f"Number of configs: {len(configs)}")

for i, config in enumerate(configs):
    print(f"Config {i+1}:")

    term = config["term"]
    print(f"  Term: {term}")

    num_names = len(config["names"])
    term_mask = term_counts["term"] == term
    assert num_names == term_counts.loc[term_mask, "count"].iloc[0]

    print(f"  Number of names: {num_names}")
    print(f"  Names: {', '.join(config['names'])}\n")

Number of configs: 3
Config 1:
  Term: short
  Number of names: 152
  Names: bull, cmip6_1885, era5_1991, SHMETRO, era5_2006, BEIJING_SUBWAY_30MIN, gfc12_load, buildings_900k, london_smart_meters_with_missing, residential_pv_power, PEMS_BAY, wind_power, cmip6_1975, elecdemand, wiki-rolling_nips, spain, cdc_fluview_who_nrevss, covid_mobility, monash_m3_quarterly, era5_2015, alibaba_cluster_trace_2018, cmip6_1930, uber_tlc_hourly, era5_2012, cmip6_1940, bdg-2_rat, era5_2018, largest_2020, cmip6_1875, cmip6_1905, borg_cluster_data_2011, tourism_yearly, traffic_weekly, largest_2018, PEMS07, era5_2001, cif_2016_6, era5_1996, bdg-2_panther, cmip6_1985, monash_m3_yearly, bitcoin_with_missing, era5_1998, era5_1992, favorita_sales, cmip6_1965, hog, era5_2005, cmip6_1850, PEMS03, cmip6_1920, covid19_energy, LOS_LOOP, cmip6_1895, tourism_quarterly, extended_web_traffic_with_missing, subseasonal_precip, cmip6_2005, era5_2016, azure_vm_traces_2017, taxi_30min, favorita_transactions, smart, kdd2022,

### Domain-Specific

View all of the unique domain-term combinations and count the number of datasets that belong to each domain-term combination.

In [82]:
domains = df["domain"].unique()
terms = df["term"].unique()

domain_term_counts = get_count_df(
    df,
    columns=["domain", "term"],
    ascending=False,
)

# Convert each forecasting term into its own column
pivoted_df = domain_term_counts.pivot(
    index="domain",
    columns="term",
    values="count",
).reset_index()

# Remove old "term" column
pivoted_df.columns.name = None

# Reorder columns
pivoted_df = pivoted_df[
    [
        "domain",
        "short",
        "medium",
        "long",
    ]
]

print(f'Domains: {", ".join(domains)}')
print(f'Terms: {", ".join(terms)}')
print(f"Number of unique domain-term combinations: {len(domain_term_counts)}")
display(pivoted_df)

Domains: Energy, Climate, Transport, Web, Healthcare, Econ/Fin, CloudOps, Sales, Nature
Terms: short, medium, long
Number of unique domain-term combinations: 27


,domain,short,medium,long
0,Climate,67,67,67
1,CloudOps,3,3,3
2,Econ/Fin,17,17,17
3,Energy,28,28,28
4,Healthcare,3,3,3
5,Nature,3,3,3
6,Sales,3,3,3
7,Transport,25,25,25
8,Web,3,3,3


Add each domain-term combination's dataset names and terms to the yaml file. We'll only create short groups for the following domains because they don't have any medium or long datasets in the train-test split:
- Econ/Fin
- Healthcare
- Sales

In [83]:
excluded_domains = [
    "Econ/Fin",
    "Healthcare",
    "Sales",
]

num_domain_configs = 0

for domain in domains:
    for term in terms:
        if domain in excluded_domains and term != "short":
            print(f"Skipping '{term}' for '{domain}'...")
            continue

        num_domain_configs += 1

        domain_mask = df["domain"] == domain
        term_mask = df["term"] == term
        filtered_df = df[domain_mask & term_mask]

        names = filtered_df["name"].tolist()

        if not names:
            print(f"No datasets found for domain '{domain}' and term '{term}'")
            continue

        data.append(
            {
                "domain": domain,
                "term": term,
                "names": names,
            }
        )

print(f"Number of domain configs: {num_domain_configs}")
with open(yaml_path, "w") as file:
    yaml.dump(data, file, sort_keys=False)

Skipping 'medium' for 'Healthcare'...
Skipping 'long' for 'Healthcare'...
Skipping 'medium' for 'Econ/Fin'...
Skipping 'long' for 'Econ/Fin'...
Skipping 'medium' for 'Sales'...
Skipping 'long' for 'Sales'...
Number of domain configs: 21


Load the YAML file to ensure the configurations were saved correctly.

In [84]:
with open(yaml_path, "r") as file:
    configs = yaml.safe_load(file)

print(f"Number of configs: {len(configs)}")

assert len(configs) == num_general_configs + num_domain_configs

for i, config in enumerate(configs):
    if "domain" not in config:
        continue

    print(f"Config {i+1}:")

    domain = config.get("domain", "N/A")
    print(f"  Domain: {domain}")

    term = config["term"]
    print(f"  Term: {term}")

    num_names = len(config["names"])
    term_mask = domain_term_counts["term"] == term
    domain_mask = domain_term_counts["domain"] == domain

    print(f"  Number of names: {num_names}")
    assert num_names == domain_term_counts.loc[term_mask & domain_mask, "count"].iloc[0]
    print(f"  Names: {', '.join(config['names'])}\n")

Number of configs: 24
Config 4:
  Domain: Energy
  Term: short
  Number of names: 28
  Names: bull, gfc12_load, buildings_900k, london_smart_meters_with_missing, residential_pv_power, wind_power, elecdemand, spain, bdg-2_rat, bdg-2_panther, hog, smart, kdd2022, gfc17_load, borealis, ideal, residential_load_power, elf, cockatoo, wind_farms_with_missing, gfc14_load, bdg-2_bear, australian_electricity_demand, pdb, lcl, sceaux, solar_power, bdg-2_fox

Config 5:
  Domain: Energy
  Term: medium
  Number of names: 28
  Names: bull, gfc12_load, buildings_900k, london_smart_meters_with_missing, residential_pv_power, wind_power, elecdemand, spain, bdg-2_rat, bdg-2_panther, hog, smart, kdd2022, gfc17_load, borealis, ideal, residential_load_power, elf, cockatoo, wind_farms_with_missing, gfc14_load, bdg-2_bear, australian_electricity_demand, pdb, lcl, sceaux, solar_power, bdg-2_fox

Config 6:
  Domain: Energy
  Term: long
  Number of names: 28
  Names: bull, gfc12_load, buildings_900k, london_smart

## Train-Test Split

Load the train-test split's metadata.

In [129]:
split = "train_test"
metadata_path = Path("resources") / split / "metadata.csv"

df = pd.read_csv(metadata_path)

print(f"Total number of {split} datasets: {len(df)}")
df.head()

Total number of train_test datasets: 97


,name,term,freq,domain,num_series,target_dim,_min_series_length,sum_series_length,prediction_length,windows
0,LOOP_SEATTLE/5T,short,5T,Transport,323,1,105120,33953760,48,20
1,LOOP_SEATTLE/D,short,D,Transport,323,1,365,117895,30,2
2,LOOP_SEATTLE/H,short,H,Transport,323,1,8760,2829480,48,19
3,M_DENSE/D,short,D,Transport,30,1,730,21900,30,3
4,M_DENSE/H,short,H,Transport,30,1,17520,525600,48,20


Create a list to store each model's dataset configuration. Later, this list will be used to create a YAML dump.

In [130]:
data = []

### General

View all of the unique forecasting terms and count the number of datasets that belong to each term. Each forecasting term multiplies the original prediciton length by a given multipler:
- "long" multiplies the original prediction length by 15
- "medium" multiplies the original prediction length by 10  
- "short" multiplies the original prediction length by 1 (no change)

In [131]:
term_counts = get_count_df(df, columns=["term"], ascending=False)

print(f"Number of unique terms: {len(term_counts)}")
display(term_counts)

Number of unique terms: 3


,term,count
0,short,55
1,long,21
2,medium,21


For each term, add all of the dataset names and the given term to our YAML data and save the resulting YAML file.

In [132]:
num_general_configs = 0
for term in df["term"].unique():
    num_general_configs += 1
    names = df[df["term"] == term]["name"].tolist()
    data.append({"term": term, "names": names})

yaml_path = Path("resources") / split / "dataset_configs.yaml"

print(f"Number of general configurations: {num_general_configs}")
with open(yaml_path, "w") as file:
    yaml.dump(data, file, sort_keys=False)

Number of general configurations: 3


Load the YAML file to ensure the configurations were saved correctly.

In [133]:
with open(yaml_path, "r") as file:
    configs = yaml.safe_load(file)

print(f"Number of configs: {len(configs)}")

for i, config in enumerate(configs):
    print(f"Config {i+1}:")

    term = config["term"]
    print(f"  Term: {term}")

    num_names = len(config["names"])
    term_mask = term_counts["term"] == term
    assert num_names == term_counts.loc[term_mask, "count"].iloc[0]

    print(f"  Number of names: {num_names}")
    print(f"  Names: {', '.join(config['names'])}\n")

Number of configs: 3
Config 1:
  Term: short
  Number of names: 55
  Names: LOOP_SEATTLE/5T, LOOP_SEATTLE/D, LOOP_SEATTLE/H, M_DENSE/D, M_DENSE/H, SZ_TAXI/15T, SZ_TAXI/H, bitbrains_fast_storage/5T, bitbrains_fast_storage/H, bitbrains_rnd/5T, bitbrains_rnd/H, bizitobs_application, bizitobs_l2c/5T, bizitobs_l2c/H, bizitobs_service, car_parts_with_missing, covid_deaths, electricity/15T, electricity/D, electricity/H, electricity/W, ett1/15T, ett1/D, ett1/H, ett1/W, ett2/15T, ett2/D, ett2/H, ett2/W, hierarchical_sales/D, hierarchical_sales/W, hospital, jena_weather/10T, jena_weather/D, jena_weather/H, kdd_cup_2018_with_missing/D, kdd_cup_2018_with_missing/H, m4_daily, m4_hourly, m4_monthly, m4_quarterly, m4_weekly, m4_yearly, restaurant, saugeenday/D, saugeenday/M, saugeenday/W, solar/10T, solar/D, solar/H, solar/W, temperature_rain_with_missing, us_births/D, us_births/M, us_births/W

Config 2:
  Term: medium
  Number of names: 21
  Names: LOOP_SEATTLE/5T, LOOP_SEATTLE/H, M_DENSE/H, SZ_TAXI

### Domain-Specific

View all of the unique domain-term combinations and count the number of datasets that belong to each domain-term combination.
- **Note:** Econ/Fin, Healthcare, and Sales only have short term datasets

In [134]:
domains = df["domain"].unique()
terms = df["term"].unique()


domain_term_counts = get_count_df(
    df,
    columns=["domain", "term"],
    ascending=False,
)

# Convert each forecasting term into its own column
pivoted_df = (
    domain_term_counts.pivot(
        index="domain",
        columns="term",
        values="count",
    )
    .fillna(0)
    .reset_index()
)

# Remove old "term" column
pivoted_df.columns.name = None

# Reorder columns
pivoted_df = pivoted_df[
    [
        "domain",
        "short",
        "medium",
        "long",
    ]
]

print(f"Number of unique domain-term combinations: {len(domain_term_counts)}")
display(pivoted_df)

Number of unique domain-term combinations: 15


,domain,short,medium,long
0,Econ/Fin,6.0,0.0,0.0
1,Energy,16.0,8.0,8.0
2,Healthcare,5.0,0.0,0.0
3,Nature,9.0,3.0,3.0
4,Sales,4.0,0.0,0.0
5,Transport,7.0,4.0,4.0
6,Web/CloudOps,8.0,6.0,6.0


Add each domain-term combination's dataset names and terms to the yaml file. We'll only create short groups for the following domains because they don't have any medium or long datasets in the train-test split:
- Econ/Fin
- Healthcare
- Sales

In [135]:
excluded_domains = [
    "Econ/Fin",
    "Healthcare",
    "Sales",
]

num_domain_configs = 0

for domain in domains:
    for term in terms:
        if domain in excluded_domains and term != "short":
            print(f"Skipping '{term}' for '{domain}'...")
            continue

        num_domain_configs += 1

        domain_mask = df["domain"] == domain
        term_mask = df["term"] == term
        filtered_df = df[domain_mask & term_mask]

        names = filtered_df["name"].tolist()

        if not names:
            print(f"No datasets found for domain '{domain}' and term '{term}'")
            continue

        data.append(
            {
                "domain": domain,
                "term": term,
                "names": names,
            }
        )

print(f"Number of domain configs: {num_domain_configs}")
with open(yaml_path, "w") as file:
    yaml.dump(data, file, sort_keys=False)

Skipping 'medium' for 'Sales'...
Skipping 'long' for 'Sales'...
Skipping 'medium' for 'Healthcare'...
Skipping 'long' for 'Healthcare'...
Skipping 'medium' for 'Econ/Fin'...
Skipping 'long' for 'Econ/Fin'...
Number of domain configs: 15


Load the YAML file to ensure the configurations were saved correctly.

In [136]:
with open(yaml_path, "r") as file:
    configs = yaml.safe_load(file)

print(f"Number of configs: {len(configs)}")

assert len(configs) == num_general_configs + num_domain_configs

for i, config in enumerate(configs):
    if "domain" not in config:
        continue

    print(f"Config {i+1}:")

    domain = config.get("domain", "N/A")
    print(f"  Domain: {domain}")

    term = config["term"]
    print(f"  Term: {term}")

    num_names = len(config["names"])
    term_mask = domain_term_counts["term"] == term
    domain_mask = domain_term_counts["domain"] == domain

    print(f"  Number of names: {num_names}")
    assert num_names == domain_term_counts.loc[term_mask & domain_mask, "count"].iloc[0]
    print(f"  Names: {', '.join(config['names'])}\n")

Number of configs: 18
Config 4:
  Domain: Transport
  Term: short
  Number of names: 7
  Names: LOOP_SEATTLE/5T, LOOP_SEATTLE/D, LOOP_SEATTLE/H, M_DENSE/D, M_DENSE/H, SZ_TAXI/15T, SZ_TAXI/H

Config 5:
  Domain: Transport
  Term: medium
  Number of names: 4
  Names: LOOP_SEATTLE/5T, LOOP_SEATTLE/H, M_DENSE/H, SZ_TAXI/15T

Config 6:
  Domain: Transport
  Term: long
  Number of names: 4
  Names: LOOP_SEATTLE/5T, LOOP_SEATTLE/H, M_DENSE/H, SZ_TAXI/15T

Config 7:
  Domain: Web/CloudOps
  Term: short
  Number of names: 8
  Names: bitbrains_fast_storage/5T, bitbrains_fast_storage/H, bitbrains_rnd/5T, bitbrains_rnd/H, bizitobs_application, bizitobs_l2c/5T, bizitobs_l2c/H, bizitobs_service

Config 8:
  Domain: Web/CloudOps
  Term: medium
  Number of names: 6
  Names: bitbrains_fast_storage/5T, bitbrains_rnd/5T, bizitobs_application, bizitobs_l2c/5T, bizitobs_l2c/H, bizitobs_service

Config 9:
  Domain: Web/CloudOps
  Term: long
  Number of names: 6
  Names: bitbrains_fast_storage/5T, bitbrains_

### Dataset-Specific 

View some of the dataset name-term combinations. For the sake of brevity, we'll only display a few of the dataset name-term combinations

In [137]:
name_term_combinations = df[["name", "term"]]

print(f"Number of unique name-term pairs: {len(name_term_combinations)}")
name_term_combinations.head()

Number of unique name-term pairs: 97


,name,term
0,LOOP_SEATTLE/5T,short
1,LOOP_SEATTLE/D,short
2,LOOP_SEATTLE/H,short
3,M_DENSE/D,short
4,M_DENSE/H,short


Add each unique name-term combination to our YAML data and save the resulting YAML file.

In [138]:
num_dataset_specific_configs = 0

for _, row in df.iterrows():
    num_dataset_specific_configs += 1
    data.append(
        {
            "term": row["term"],
            "names": [row["name"]],
        }
    )

print(f"Number of dataset-specific configurations: {num_dataset_specific_configs}")
with open(yaml_path, "w") as file:
    yaml.dump(data, file, sort_keys=False)

Number of dataset-specific configurations: 97


Load the YAML file to ensure the configurations were saved correctly.

In [139]:
with open(yaml_path, "r") as file:
    configs = yaml.safe_load(file)

assert (
    len(configs)
    == num_general_configs + num_domain_configs + num_dataset_specific_configs
)

for i, config in enumerate(configs):
    print(f"Config {i+1}:")

    term = config["term"]
    print(f"  Term: {term}")

    num_names = len(config["names"])
    term_mask = term_counts["term"] == term

    print(f"  Number of names: {num_names}")
    print(f"  Names: {', '.join(config['names'])}\n")

Config 1:
  Term: short
  Number of names: 55
  Names: LOOP_SEATTLE/5T, LOOP_SEATTLE/D, LOOP_SEATTLE/H, M_DENSE/D, M_DENSE/H, SZ_TAXI/15T, SZ_TAXI/H, bitbrains_fast_storage/5T, bitbrains_fast_storage/H, bitbrains_rnd/5T, bitbrains_rnd/H, bizitobs_application, bizitobs_l2c/5T, bizitobs_l2c/H, bizitobs_service, car_parts_with_missing, covid_deaths, electricity/15T, electricity/D, electricity/H, electricity/W, ett1/15T, ett1/D, ett1/H, ett1/W, ett2/15T, ett2/D, ett2/H, ett2/W, hierarchical_sales/D, hierarchical_sales/W, hospital, jena_weather/10T, jena_weather/D, jena_weather/H, kdd_cup_2018_with_missing/D, kdd_cup_2018_with_missing/H, m4_daily, m4_hourly, m4_monthly, m4_quarterly, m4_weekly, m4_yearly, restaurant, saugeenday/D, saugeenday/M, saugeenday/W, solar/10T, solar/D, solar/H, solar/W, temperature_rain_with_missing, us_births/D, us_births/M, us_births/W

Config 2:
  Term: medium
  Number of names: 21
  Names: LOOP_SEATTLE/5T, LOOP_SEATTLE/H, M_DENSE/H, SZ_TAXI/15T, bitbrains_fast_